# Restaurant Sales Analysis Project
### NoteBook 2: Q2_Data_Cleaning

## 1. Import Libraries

In [1]:
import pandas as pd

## 2. Load Data


In [2]:
df2 = pd.read_excel("../../../Data/raw/Q2.xlsx")

## 3. Initial Data Inspection

This section presents an initial assessment of the dataset, including its structure, data types, missing values, and records with zero values in key financial fields to identify potential data quality issues before the cleaning process.

### 3.1 Verify Dataset Structure

In [3]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45334 entries, 0 to 45333
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Date                    15366 non-null  datetime64[ns]
 1   Receipt Number          15366 non-null  object        
 2   Customer                15357 non-null  object        
 3   Invoice                 15353 non-null  object        
 4   Is Refunded             15366 non-null  float64       
 5   Order Lines/Product     45334 non-null  object        
 6   Order Lines/Quantity    45334 non-null  float64       
 7   Order Lines/Unit Price  45334 non-null  float64       
 8   Order Lines/Subtotal    45334 non-null  float64       
 9   Total                   15366 non-null  float64       
dtypes: datetime64[ns](1), float64(5), object(4)
memory usage: 3.5+ MB


### 3.2 Inspect Missing Values

In [4]:
df2.isna().sum()

Date                      29968
Receipt Number            29968
Customer                  29977
Invoice                   29981
Is Refunded               29968
Order Lines/Product           0
Order Lines/Quantity          0
Order Lines/Unit Price        0
Order Lines/Subtotal          0
Total                     29968
dtype: int64

### 3.3 Inspect transaction records for zero values in key financial fields.

In [5]:
# Count transaction records containing zero values in key financial fields.
zero_value_records = df2[
    (df2["Order Lines/Quantity"] == 0) |
    (df2["Order Lines/Subtotal"] == 0) |
    (df2["Total"] == 0) |
    (df2["Order Lines/Unit Price"] == 0)
]

print(f"Number of records requiring review: {len(zero_value_records)}")

Number of records requiring review: 91


### 3.4 Check for receipt numbers linked to multiple transaction dates.

In [6]:
df2.groupby("Receipt Number")["Date"].nunique().sort_values(ascending=False).head(23)

Receipt Number
طلب 08611-002-2934     2
طلب 08248-001-3859     2
طلب 08300-002-2055     2
طلب 05595-002-5249     2
طلب 08436-002-0026     2
طلب 05595-002-5527     2
طلب 06778-002-12886    2
طلب 08611-002-0263     2
طلب 08981-002-0658     2
طلب 08659-002-2459     2
طلب 07772-008-26221    2
طلب 06988-001-15038    2
طلب 07736-003-3889     2
طلب 07503-004-24860    2
طلب 08611-002-3862     2
طلب 07811-002-1601     2
طلب 08233-002-0069     2
طلب 08436-002-3262     2
طلب 08436-002-3113     2
طلب 08384-002-3920     2
طلب 07020-005-12594    2
طلب 08147-009-23452    2
طلب 07819-001-21285    1
Name: Date, dtype: int64

## 4. Rename Columns

Column names were renamed to improve readability and simplify the analysis.

In [7]:
df2.rename(columns={
    "Date": "Date",
    "Receipt Number": "Receipt Number",
    "Customer": "Customer",
    "Invoice": "Invoice",
    "Is Refunded": "Is Refunded", 
    "Order Lines/Product": "Product",
    "Order Lines/Quantity": "Quantity",
    "Order Lines/Unit Price": "Unit_Price",
    "Order Lines/Subtotal": "Subtotal",
    "Total": "Total"
}, inplace=True)

## 5. Data Cleaning, Preprocessing, and Data Consistency Checks

This section performs the main data cleaning and preprocessing steps to improve the quality and consistency of the dataset. First, a copy of the original dataset is created to preserve the raw data. Unnecessary columns (`Invoice` and `Is Refunded`) are removed as they are not required for the analysis.

Next, forward filling is applied only to selected columns (`Date`, `Receipt Number`, `Quantity`, `Unit_Price`, and `Subtotal`) because these values are expected to remain consistent within the same transaction. The `Customer` column is intentionally excluded to avoid assigning incorrect customer names to unrelated transactions, while the `Total` column is excluded because it represents the overall transaction amount rather than individual product lines, and propagating its values could introduce inaccurate financial information. Records with a `Subtotal` value of zero are then removed because they do not represent valid sales line items.

A data consistency check is then performed to identify receipt numbers associated with multiple transaction dates. The inspection revealed a small number of duplicate receipt numbers linked to different transaction times. These duplicate cases were resolved by removing the oldest transaction record for each affected receipt number while retaining the most recent one. Finally, the dataset was inspected for fully duplicated records to ensure data integrity. Any exact duplicate records were identified and removed to eliminate redundant transaction entries. A final validation check was then performed to confirm that no duplicate records remained before proceeding to the exploratory data analysis.

In [8]:
# Create a copy of the original dataset to preserve the raw data.
df2_copy = df2.copy()
df2_copy = df2_copy.drop(columns=['Invoice', 'Is Refunded'])

In [9]:
# Propagate transaction-level values to product rows within the same transaction.
columns_to_fill = [
    "Date",
    "Receipt Number",
    "Quantity",
    "Unit_Price",
    "Subtotal",

]


df2_copy[columns_to_fill] = df2_copy[columns_to_fill].ffill()

In [10]:
df2_copy = df2_copy[df2_copy['Subtotal'] != 0]

In [11]:
df2_copy.isna().sum()

Date                  0
Receipt Number        0
Customer          29897
Product               0
Quantity              0
Unit_Price            0
Subtotal              0
Total             29888
dtype: int64

In [12]:
# Function Remove the oldest transaction (based on Date) for a given Receipt Number.
def remove_oldest_receipt_record(df, receipt_number):

    # Check if the receipt number exists
    if receipt_number not in df["Receipt Number"].values:
        print(f"Receipt Number '{receipt_number}' not found.")
        return df

    # Find the oldest date for the receipt
    old_date = df.loc[
        df["Receipt Number"] == receipt_number,
        "Date"
    ].min()

    # Remove all rows belonging to the oldest transaction
    df = df[
        ~(
            (df["Receipt Number"] == receipt_number) &
            (df["Date"] == old_date)
        )
    ]

    return df

In [13]:
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 08611-002-2934")
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 08248-001-3859")
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 08300-002-2055")
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 08436-002-0026")
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 08611-002-0263")
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 08981-002-0658")
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 08659-002-2459")
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 07772-008-26221")
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 07736-003-3889")
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 07503-004-24860")
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 08611-002-3862")
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 07811-002-1601")
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 07811-002-1601")
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 08233-002-0069")
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 08436-002-3262")
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 08436-002-3113")
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 07020-005-12594")
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 07020-005-12594")
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 08147-009-23452")
df2_copy = remove_oldest_receipt_record(df2_copy, "طلب 07819-001-21285")

In [14]:
# Check for duplicate records.
print(f"Duplicate records: {df2_copy.duplicated().sum()}")

Duplicate records: 47


In [15]:
# Remove duplicate records.
df2_copy = df2_copy.drop_duplicates()

In [16]:
# Verify that no duplicate records remain.
print(f"Duplicate records after removal: {df2_copy.duplicated().sum()}")

Duplicate records after removal: 0


## 6. Export Clean Dataset

In [17]:
df2_copy.to_excel("../../../Data/Cleaned/clean_q2_data.xlsx", index=False)